# Gerar Parquet Censo Escolar

Notebook simplificado para gerar `trabalho/censo-escolar.parquet` com leitura em chunks e baixo uso de memória.

## 1) Configuração

### 1.1 Imports e caminhos

In [ ]:
from pathlib import Path
import csv
import re

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'datasets').exists() and (PROJECT_ROOT.parent / 'datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASETS_DIR = PROJECT_ROOT / 'datasets'
TRABALHO_DIR = PROJECT_ROOT / 'trabalho'
OUTPUT_PATH = TRABALHO_DIR / 'censo-escolar.parquet'

YEARS = range(1995, 2026)
CHUNKSIZE = 50_000
MAX_OUTPUT_COLUMNS = 100
MAX_SOURCE_COLUMNS = 300

TRABALHO_DIR.mkdir(parents=True, exist_ok=True)

print(f'Datasets: {DATASETS_DIR}')
print(f'Saída: {OUTPUT_PATH}')


Datasets: /home/gabriel/ciencias-computacao/programacao-para-analise-de-dados/datasets
Saída: /home/gabriel/ciencias-computacao/programacao-para-analise-de-dados/trabalho/censo-escolar.parquet


## 2) Especificação das colunas

### 2.1 Mapa de aliases

In [5]:
def spec(kind, aliases):
    return {'kind': kind, 'aliases': aliases}

COLUMN_SPECS = {
    'ano_censo': spec('int', ['NU_ANO_CENSO', 'ANO', 'NU_ANO']),
    'id_escola': spec('string', ['CO_ENTIDADE', 'MASCARA']),
    'co_municipio': spec('string', ['CO_MUNICIPIO', 'CODMUNIC', 'CO_IBGE']),
    'no_municipio': spec('string', ['NO_MUNICIPIO', 'MUNIC']),
    'sg_uf': spec('string', ['SG_UF', 'SIGLA']),
    'co_uf': spec('int', ['CO_UF']),
    'no_uf': spec('string', ['NO_UF', 'UF']),
    'no_regiao': spec('string', ['NO_REGIAO']),
    'co_regiao': spec('int', ['CO_REGIAO']),
    'no_entidade': spec('string', ['NO_ENTIDADE']),
    'dependencia_administrativa': spec('category_dependencia', ['TP_DEPENDENCIA', 'DEP']),
    'localizacao': spec('category_localizacao', ['TP_LOCALIZACAO', 'LOC']),
    'situacao_funcionamento': spec('category_situacao', ['TP_SITUACAO_FUNCIONAMENTO', 'CODFUNC']),
    'tp_localizacao_diferenciada': spec('int', ['TP_LOCALIZACAO_DIFERENCIADA']),
    'tp_categoria_escola_privada': spec('int', ['TP_CATEGORIA_ESCOLA_PRIVADA']),

    'in_regular': spec('indicator', ['IN_REGULAR', 'ENSREGULAR', 'NIVELCRE', 'NIVELPRE', 'NIV_1GRAU', 'NIV_2GRAU', 'NIV_F1A4_8', 'NIV_F5A8_8', 'NIVELMED']),
    'in_creche': spec('indicator', ['IN_INF_CRE', 'IN_COMUM_CRECHE', 'IN_ESP_EXCLUSIVA_CRECHE', 'NIVELCRE']),
    'in_pre_escola': spec('indicator', ['IN_INF_PRE', 'IN_COMUM_PRE', 'IN_ESP_EXCLUSIVA_PRE', 'NIVELPRE']),
    'in_fundamental': spec('indicator', ['IN_FUND', 'IN_FUND_AI', 'IN_FUND_AF', 'IN_COMUM_FUND_AI', 'IN_COMUM_FUND_AF', 'NIV_1GRAU', 'NIV_F1A4_8', 'NIV_F5A8_8', 'NIV_F1A4', 'NIV_F5A8']),
    'in_medio': spec('indicator', ['IN_MED', 'IN_COMUM_MEDIO_MEDIO', 'IN_COMUM_MEDIO_INTEGRADO', 'NIV_2GRAU', 'NIVELMED', 'NIVELMEDIO']),
    'in_eja': spec('indicator', ['IN_EJA', 'IN_COMUM_EJA_FUND', 'IN_COMUM_EJA_MEDIO', 'ENSSUPLET', 'SUPL_AVA', 'SUPL_SAVA']),
    'in_profissionalizante': spec('indicator', ['IN_PROFISSIONALIZANTE', 'IN_COMUM_PROF', 'IN_PROF', 'IN_PROF_TEC', 'EDPROFIS']),
    'in_especial': spec('indicator', ['IN_ESPECIAL_EXCLUSIVA', 'IN_ESP', 'IN_ESP_CC', 'IN_ESP_CE', 'ESP_EXCL', 'ESP_T_ES', 'ENS_INCL']),
    'in_educacao_indigena': spec('indicator', ['IN_EDUCACAO_INDIGENA', 'ED_INDIG']),

    'in_predio_escolar': spec('indicator', ['IN_LOCAL_FUNC_PREDIO_ESCOLAR', 'PRED_ESC']),
    'in_predio_compartilhado': spec('indicator', ['IN_PREDIO_COMPARTILHADO', 'PRED_COM']),
    'in_agua_potavel': spec('indicator', ['IN_AGUA_POTAVEL', 'AGUA_FIL']),
    'in_agua_rede_publica': spec('indicator', ['IN_AGUA_REDE_PUBLICA', 'AGUA_PUB']),
    'in_agua_poco_artesiano': spec('indicator', ['IN_AGUA_POCO_ARTESIANO', 'AGUA_ART']),
    'in_agua_inexistente': spec('indicator', ['IN_AGUA_INEXISTENTE', 'AGUA_INE']),
    'in_energia_rede_publica': spec('indicator', ['IN_ENERGIA_REDE_PUBLICA', 'ENER_PUB']),
    'in_energia_gerador': spec('indicator', ['IN_ENERGIA_GERADOR_FOSSIL', 'IN_ENERGIA_GERADOR', 'ENER_GER']),
    'in_energia_inexistente': spec('indicator', ['IN_ENERGIA_INEXISTENTE', 'ENER_INE']),

    'qt_mat_bas': spec('int', ['QT_MAT_BAS']),
    'qt_mat_fund': spec('int', ['QT_MAT_FUND']),
    'qt_mat_med': spec('int', ['QT_MAT_MED']),
    'qt_doc_bas': spec('int', ['QT_DOC_BAS']),
    'qt_doc_fund': spec('int', ['QT_DOC_FUND']),
    'qt_doc_med': spec('int', ['QT_DOC_MED']),
    'qt_tur_bas': spec('int', ['QT_TUR_BAS']),
    'qt_tur_fund': spec('int', ['QT_TUR_FUND']),
    'qt_tur_med': spec('int', ['QT_TUR_MED']),
}

OUTPUT_COLUMNS = list(COLUMN_SPECS)[:MAX_OUTPUT_COLUMNS]
print(f'Colunas finais: {len(OUTPUT_COLUMNS)}')


Colunas finais: 42


## 3) Descoberta dos arquivos

### 3.1 Planejamento anual (arquivo, separador, encoding e colunas fonte)

In [6]:
def extract_year(path):
    match = re.search(r'(?:19|20)\d{2}', str(path))
    return int(match.group(0)) if match else None

def select_primary_file(paths, year):
    if year <= 2006:
        expected = f'censoesc_{year}.csv'
    elif year == 2025:
        expected = f'tabela_escola_{year}.csv'
    else:
        expected = f'microdados_ed_basica_{year}.csv'

    for path in paths:
        if path.name.lower() == expected:
            return path
    raise FileNotFoundError(f'Arquivo principal não encontrado para {year}: {expected}')

def sniff_csv(path):
    forced_pipe = bool(re.search(r'^(CENSOESC|EDUCPROF|INDIC|MEDPROF|EM\d+|ES\d+)', path.name, re.I))
    encodings = ['latin1', 'utf-8-sig'] if forced_pipe else ['utf-8-sig', 'latin1']

    for encoding in encodings:
        try:
            if forced_pipe:
                cols = pd.read_csv(path, sep='|', encoding=encoding, encoding_errors='replace', nrows=0, engine='python').columns.tolist()
                return {'sep': '|', 'encoding': encoding, 'columns': cols}

            sample = path.read_text(encoding='latin1', errors='replace')[:50_000]
            first_line = sample.splitlines()[0] if sample else ''
            counts = {sep: first_line.count(sep) for sep in [';', '|', '\t', ',']}
            sep = max(counts, key=counts.get)
            if counts[sep] == 0:
                sep = csv.Sniffer().sniff(sample, delimiters=';,|\t,').delimiter

            cols = pd.read_csv(path, sep=sep, encoding=encoding, encoding_errors='replace', nrows=0, engine='python').columns.tolist()
            return {'sep': sep, 'encoding': encoding, 'columns': cols}
        except Exception:
            continue

    raise ValueError(f'Não foi possível ler o cabeçalho de {path}')

csv_files_by_year = {year: [] for year in YEARS}
for path in sorted(DATASETS_DIR.rglob('*')):
    if path.suffix.lower() != '.csv':
        continue
    year = extract_year(path)
    if year in csv_files_by_year:
        csv_files_by_year[year].append(path)

plan = []
for year in YEARS:
    primary = select_primary_file(csv_files_by_year[year], year)
    csv_spec = sniff_csv(primary)
    available = set(csv_spec['columns'])

    source_columns = []
    for target in OUTPUT_COLUMNS:
        for alias in COLUMN_SPECS[target]['aliases']:
            if alias in available and alias not in source_columns:
                source_columns.append(alias)

    assert len(source_columns) <= MAX_SOURCE_COLUMNS, (year, len(source_columns))

    plan.append({
        'year': year,
        'path': primary,
        'sep': csv_spec['sep'],
        'encoding': csv_spec['encoding'],
        'columns': csv_spec['columns'],
        'source_columns': source_columns,
    })

pd.DataFrame([{
    'ano': item['year'],
    'arquivo': item['path'].name,
    'sep': item['sep'],
    'encoding': item['encoding'],
    'colunas_originais': len(item['columns']),
    'colunas_lidas': len(item['source_columns']),
} for item in plan])


,ano,arquivo,sep,encoding,colunas_originais,colunas_lidas
0,1995,CENSOESC_1995.CSV,|,latin1,479,13
1,1996,CENSOESC_1996.CSV,|,latin1,812,15
2,1997,CENSOESC_1997.CSV,|,latin1,367,24
3,1998,CENSOESC_1998.CSV,|,latin1,893,25
4,1999,CENSOESC_1999.CSV,|,latin1,1065,26
5,2000,CENSOESC_2000.CSV,|,latin1,872,25
6,2001,CENSOESC_2001.CSV,|,latin1,1244,27
7,2002,CENSOESC_2002.CSV,|,latin1,1307,27
8,2003,CENSOESC_2003.CSV,|,latin1,1853,27
9,2004,CENSOESC_2004.CSV,|,latin1,3260,29


## 4) Normalização

### 4.1 Funções simples de limpeza e mapeamento

In [7]:
def clean_series(series):
    return (
        series.astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})
    )

def first_available_series(chunk, aliases):
    selected = [clean_series(chunk[a]) for a in aliases if a in chunk.columns]
    if not selected:
        return pd.Series(pd.NA, index=chunk.index, dtype='string')
    out = selected[0]
    for s in selected[1:]:
        out = out.fillna(s)
    return out

def int_series(chunk, aliases):
    return pd.to_numeric(first_available_series(chunk, aliases), errors='coerce').astype('Int64')

def indicator_from_series(series):
    value = clean_series(series).str.lower()
    number = pd.to_numeric(value.str.replace(',', '.', regex=False), errors='coerce')

    out = pd.Series(pd.NA, index=series.index, dtype='Int8')
    out[value.isin({'1', 's', 'sim', 'y', 'yes', 'true', 'ativo'}) | (number > 0)] = 1
    out[value.isin({'0', 'n', 'nao', 'não', 'no', 'false', 'inativo'}) | (number == 0)] = 0
    return out

def indicator_series(chunk, aliases):
    selected = [indicator_from_series(chunk[a]) for a in aliases if a in chunk.columns]
    if not selected:
        return pd.Series(pd.NA, index=chunk.index, dtype='Int8')
    out = selected[0]
    for s in selected[1:]:
        out = out.fillna(s)
        out = out.mask((out == 0) & (s == 1), 1)
    return out.astype('Int8')

def dependencia_series(chunk, aliases):
    raw = first_available_series(chunk, aliases)
    key = raw.str.lower()
    mapped = key.map({
        '1': 'Federal', 'federal': 'Federal',
        '2': 'Estadual', 'estadual': 'Estadual',
        '3': 'Municipal', 'municipal': 'Municipal',
        '4': 'Privada', 'particular': 'Privada', 'privada': 'Privada',
    })
    return mapped.astype('string').fillna(raw)

def localizacao_series(chunk, aliases):
    raw = first_available_series(chunk, aliases)
    key = raw.str.lower()
    mapped = key.map({'1': 'Urbana', 'urbana': 'Urbana', '2': 'Rural', 'rural': 'Rural'})
    return mapped.astype('string').fillna(raw)

def situacao_series(chunk, aliases):
    raw = first_available_series(chunk, aliases)
    key = raw.str.lower()
    mapped = key.map({
        '1': 'Ativa', 'ativo': 'Ativa', 'em atividade': 'Ativa',
        '2': 'Paralisada', 'paralisada': 'Paralisada',
        '3': 'Extinta', 'extinta': 'Extinta',
    })
    return mapped.astype('string').fillna(raw)

def output_series(chunk, target):
    info = COLUMN_SPECS[target]
    kind = info['kind']
    aliases = info['aliases']

    if kind == 'int':
        return int_series(chunk, aliases)
    if kind == 'indicator':
        return indicator_series(chunk, aliases)
    if kind == 'category_dependencia':
        return dependencia_series(chunk, aliases)
    if kind == 'category_localizacao':
        return localizacao_series(chunk, aliases)
    if kind == 'category_situacao':
        return situacao_series(chunk, aliases)

    return first_available_series(chunk, aliases)

def normalize_chunk(chunk):
    out = pd.DataFrame(index=chunk.index)
    for col in OUTPUT_COLUMNS:
        out[col] = output_series(chunk, col)
    return out.loc[:, OUTPUT_COLUMNS]


## 5) Tabelas auxiliares de 2025

### 5.1 Carregar tabelas de matrícula, docente e turma

In [8]:
def find_2025_table(prefix):
    for p in csv_files_by_year[2025]:
        if p.name.lower().startswith(prefix.lower()):
            return p
    return None

def read_2025_aux_table(prefix):
    path = find_2025_table(prefix)
    if path is None:
        return None

    csv_spec = sniff_csv(path)
    available = set(csv_spec['columns'])

    wanted = {'NU_ANO_CENSO', 'CO_ENTIDADE'}
    for col in OUTPUT_COLUMNS:
        for alias in COLUMN_SPECS[col]['aliases']:
            if alias in available:
                wanted.add(alias)

    usecols = sorted(wanted & available)
    if len(usecols) <= 2:
        return None

    df = pd.read_csv(
        path,
        sep=csv_spec['sep'],
        encoding=csv_spec['encoding'],
        encoding_errors='replace',
        usecols=usecols,
        dtype='string',
    )

    if 'NU_ANO_CENSO' in df.columns:
        df['NU_ANO_CENSO'] = clean_series(df['NU_ANO_CENSO'])
    if 'CO_ENTIDADE' in df.columns:
        df['CO_ENTIDADE'] = clean_series(df['CO_ENTIDADE'])
    return df

aux_2025 = []
for prefix in ['Tabela_Matricula_2025', 'Tabela_Docente_2025', 'Tabela_Turma_2025']:
    df_aux = read_2025_aux_table(prefix)
    if df_aux is not None:
        aux_2025.append(df_aux)

print(f'Tabelas auxiliares 2025 carregadas: {len(aux_2025)}')


Tabelas auxiliares 2025 carregadas: 3


## 6) Escrita do parquet

### 6.1 Processamento por ano e por chunk

In [9]:
def iter_year_chunks(item):
    reader = pd.read_csv(
        item['path'],
        sep=item['sep'],
        encoding=item['encoding'],
        encoding_errors='replace',
        usecols=item['source_columns'],
        dtype='string',
        chunksize=CHUNKSIZE,
        engine='python' if item['sep'] == '|' else 'c',
        on_bad_lines='warn',
    )

    for chunk in reader:
        if item['year'] == 2025 and aux_2025:
            if 'NU_ANO_CENSO' in chunk.columns:
                chunk['NU_ANO_CENSO'] = clean_series(chunk['NU_ANO_CENSO'])
            if 'CO_ENTIDADE' in chunk.columns:
                chunk['CO_ENTIDADE'] = clean_series(chunk['CO_ENTIDADE'])

            for df_aux in aux_2025:
                chunk = chunk.merge(df_aux, on=['NU_ANO_CENSO', 'CO_ENTIDADE'], how='left', suffixes=('', '_aux'))

        yield normalize_chunk(chunk)

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

writer = None
rows_by_year = {}

try:
    for item in plan:
        year_rows = 0
        for df_chunk in iter_year_chunks(item):
            table = pa.Table.from_pandas(df_chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(OUTPUT_PATH, table.schema, compression='zstd')
            writer.write_table(table)
            year_rows += len(df_chunk)

        rows_by_year[item['year']] = year_rows
        print(f"{item['year']}: {year_rows:,} linhas")
finally:
    if writer is not None:
        writer.close()

print(f'Parquet gerado: {OUTPUT_PATH}')


1995: 243,637 linhas
1996: 276,731 linhas
1997: 273,951 linhas
1998: 267,532 linhas
1999: 266,645 linhas
2000: 261,988 linhas
2001: 264,735 linhas
2002: 256,986 linhas
2003: 253,405 linhas
2004: 248,257 linhas
2005: 248,103 linhas
2006: 241,817 linhas
2007: 198,507 linhas
2008: 205,699 linhas
2009: 203,455 linhas
2010: 200,876 linhas
2011: 242,147 linhas
2012: 242,136 linhas
2013: 242,680 linhas
2014: 242,929 linhas
2015: 237,879 linhas
2016: 237,506 linhas
2017: 236,481 linhas
2018: 236,460 linhas
2019: 228,521 linhas
2020: 224,229 linhas
2021: 221,140 linhas
2022: 224,649 linhas
2023: 217,625 linhas
2024: 215,545 linhas
2025: 214,192 linhas
Parquet gerado: /home/gabriel/ciencias-computacao/programacao-para-analise-de-dados/trabalho/censo-escolar.parquet


## 7) Validação rápida

### 7.1 Métricas do arquivo e amostra

In [10]:
parquet_file = pq.ParquetFile(OUTPUT_PATH)

ano_min = None
ano_max = None
for batch in parquet_file.iter_batches(columns=['ano_censo'], batch_size=250_000):
    year_series = batch.column(0).to_pandas()
    cur_min = year_series.min(skipna=True)
    cur_max = year_series.max(skipna=True)
    ano_min = cur_min if ano_min is None else min(ano_min, cur_min)
    ano_max = cur_max if ano_max is None else max(ano_max, cur_max)

print(f'Linhas: {parquet_file.metadata.num_rows:,}')
print(f'Colunas: {parquet_file.metadata.num_columns}')
print(f'Tamanho: {OUTPUT_PATH.stat().st_size / 1024**2:.1f} MB')
print(f'Anos: {(ano_min, ano_max)}')

next(parquet_file.iter_batches(batch_size=5)).to_pandas()


Linhas: 7,376,443
Colunas: 42
Tamanho: 131.4 MB
Anos: (np.int64(1995), np.int64(2025))


,ano_censo,id_escola,co_municipio,no_municipio,sg_uf,co_uf,no_uf,no_regiao,co_regiao,no_entidade,...,in_energia_inexistente,qt_mat_bas,qt_mat_fund,qt_mat_med,qt_doc_bas,qt_doc_fund,qt_doc_med,qt_tur_bas,qt_tur_fund,qt_tur_med
0,1995,0000000027,31070300620005,BELO HORIZONTE,MG,<NA>,Minas Gerais,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,1995,0000000035,31070300620005,BELO HORIZONTE,MG,<NA>,Minas Gerais,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,1995,0000000043,31070300620005,BELO HORIZONTE,MG,<NA>,Minas Gerais,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,1995,0000000051,31070300620005,BELO HORIZONTE,MG,<NA>,Minas Gerais,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,1995,0000000060,31070300620005,BELO HORIZONTE,MG,<NA>,Minas Gerais,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
